# 02 - Cohort Retention
**Question:** what share of customers come back, and how does retention evolve after their first order?

Source: `marts.mart_cohort_retention` (cohort = month of a customer's first delivered order, cohorts inside the analysis window only).
Close Jupyter's other connections to the warehouse before running dbt; this notebook opens it read-only.

In [ ]:
import sys
from pathlib import Path

import duckdb
import pandas as pd
import plotly.express as px

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))

from ingestion.config import get_warehouse_path  # noqa: E402

con = duckdb.connect(str(get_warehouse_path()), read_only=True)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")


def q(sql: str) -> pd.DataFrame:
    """Run SQL against the warehouse and return a DataFrame."""
    return con.execute(sql).df()

## Retention heatmap
Rows are cohorts, columns are months since the first order. Month 0 is always 100%.

In [ ]:
ret = q("SELECT * FROM marts.mart_cohort_retention")
ret["cohort"] = pd.to_datetime(ret["cohort_month"]).dt.strftime("%Y-%m")
heat = ret.pivot(index="cohort", columns="months_since_first", values="retention_pct")
# Hide month 0 (always 100%) so the interesting small values are visible
fig = px.imshow(heat.drop(columns=0), aspect="auto", color_continuous_scale="Blues",
                labels={"x": "Months since first order", "y": "Cohort", "color": "Retention %"},
                title="Retention % by cohort (month 0 hidden)")
fig.show()

## Average retention by month since first order
Weighted by cohort size, and only counting cohorts old enough to have reached that month (younger cohorts have not had time yet, so averaging them in would understate retention).

In [ ]:
avg = (
    ret.groupby("months_since_first")
    .agg(active=("active_customers", "sum"), size=("cohort_size", "sum"), cohorts=("cohort", "nunique"))
    .reset_index()
)
avg["retention_pct"] = 100 * avg["active"] / avg["size"]
display(avg)
px.line(avg[avg.months_since_first > 0], x="months_since_first", y="retention_pct", markers=True,
        title="Average retention % by months since first order").show()

## How to read this
- With one-off marketplace buyers, retention after month 0 is expected to be low. A flat line near a few percent is a finding, not a bug.
- Retention at month N only uses cohorts with at least N months of history; the right-hand side of the chart rests on fewer cohorts and is noisier.
- Record the headline number (for example month-1 and month-6 retention) in `docs/insights.md`.